# Topic: ML: Cross-Validation

## Definition (30-second explanation)
* Cross-Validation (CV) evaluates model performance by splitting a dataset into $k$ equal parts (folds).
* The model is trained $k$ times; each time, $k-1$ folds are used for training, and the remaining 1 fold is held out for testing.
* The final performance is the average score across all $k$ iterations, ensuring every single data point is used for testing exactly once.

## Why Interviewers Ask This
* **Robustness Check:** Assesses your understanding of variance in model evaluation (a single random split can be "lucky" or "unlucky").
* **Data Leakage Awareness:** Tests if you know how to properly apply preprocessing steps (like scaling) *inside* the CV loop using Pipelines.
* **Class Imbalance:** Checks if you default to `StratifiedKFold` for classification to prevent folds with zero minority class samples.

## Core Concepts
* **K-Fold CV:** The standard approach for regression, splitting data into $k$ consecutive or shuffled blocks.
* **Stratified K-Fold CV:** For classification; guarantees that the target class distribution (e.g., 90% positive, 10% negative) is preserved in every single fold.
* **Bias-Variance Trade-off in $k$:** A lower $k$ (e.g., 3) has higher bias and lower variance. A higher $k$ (e.g., 10 or Leave-One-Out) has lower bias but higher variance and computation time.
* **Stability Metric:** The standard deviation (`std`) of the CV scores tells you how stable the model is across different data subsets (lower is better).

## When to Use
* When working with small to medium datasets where you cannot afford to "waste" data on a single hold-out validation set.
* When comparing the performance of multiple algorithms or tuning hyperparameters.
* Whenever an interviewer explicitly asks for a "reliable" or "robust" model evaluation strategy.

## Advantages
* Provides a much more reliable, lower-variance estimate of real-world model performance compared to a simple train-test split.
* Maximizes data utility, as 100% of the dataset is eventually used for both training and validation.

## Limitations
* Computationally expensive: running a 10-fold CV takes roughly 10 times longer than a single train-test split.
* Not suitable for massive datasets (millions of rows) where a simple hold-out set is statistically sufficient and CV would take days.
* Standard CV breaks temporal logic; time-series data requires `TimeSeriesSplit`.

## Common Comparisons
* **Cross-Validation vs. Train-Test Split:** Train-test is fast but high-variance (dependent on the random seed). CV is slow but low-variance and highly reliable.
* **K-Fold vs. Stratified K-Fold:** K-Fold is for continuous targets (regression). Stratified is for categorical targets (classification) to maintain class balance.

## Common Interview Traps
* **The Preprocessing Leakage Trap:** Fitting a `StandardScaler` or `SimpleImputer` on the *entire* dataset before running `cross_val_score`. This leaks information from the validation folds into the training folds.
* **Ignoring Group Structures:** Using standard CV on medical data where multiple rows belong to the same patient. The same patient might end up in both train and test folds. (Fix: Use `GroupKFold`).

## Python / SQL Syntax
```python
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Define strategy and model
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = LogisticRegression()

# Execute cross-validation
scores = cross_val_score(model, X, y, cv=cv_strategy, scoring='accuracy')

print(f"Mean CV Score: {scores.mean():.4f}")
print(f"Std CV Score: {scores.std():.4f}")
```

## Important Formula
* **CV Score Average:** $CV_{score} = \frac{1}{k} \sum_{i=1}^{k} Score_i$

## 45-Second Interview Answer
"Cross-validation is a technique to robustly evaluate model performance by splitting data into $k$ folds, training on $k-1$, and testing on the remaining fold, repeating this until every fold has been a test set. It provides a more reliable, lower-variance metric than a single train-test split. For classification tasks, I always use `StratifiedKFold` to maintain class balance. Most importantly, I wrap my preprocessing steps and model in a scikit-learn `Pipeline` before passing it to the CV function to absolutely guarantee no data leakage occurs across folds."